In [0]:
1. Column Level redaction 
2. Row Level Security
3. Data Masking

In [0]:
create or replace view dev.naval_silver.customers_detials as 
select customer_id, customer_name, date_of_birth, email,member_since, telephone from dev.naval_silver.customers_cleaned

In [0]:
grant use catalog on catalog dev to `account users`;
grant use schema on schema dev.naval_silver to `account users`;
grant select on table dev.naval_silver.customers_detials to `account users`

In [0]:
select * from dev.naval_silver.customers_detials

customer_id,customer_name,date_of_birth,email,member_since,telephone
9179,Richard Cox,1996-10-25,REDACTED,2024-09-26,+1 6680703335
4858,Carla Morton,2004-06-21,REDACTED,2024-09-15,+1 8616454195
7207,Billy Scott,1997-03-17,REDACTED,2024-09-23,+1 5544387564
8539,Lori Mason,2002-11-01,REDACTED,2024-09-12,+1 0498301620
9706,Jennifer Haas,2001-04-03,REDACTED,2024-10-05,+1 4725460000
9263,Joseph Keller,2003-02-11,REDACTED,2024-10-04,+1 3817867756
5028,Jessica Harris,2004-04-19,REDACTED,2024-09-10,+1 8604009935
9018,William Carter,2003-09-05,REDACTED,2024-10-08,+1 1448753611
8580,Shannon Austin,2002-03-22,REDACTED,2024-10-07,+1 4594705629
3409,Andrew Phillips,2003-04-17,REDACTED,2024-09-30,+1 4079273853


Column Level

In [0]:
create or replace view dev.naval_silver.customers_detials as 
select 
customer_id, 
customer_name, 
date_of_birth, 
CASE WHEN
    is_account_group_member('dataeng') THEN 'REDACTED'
    ELSE email
  END AS email,
member_since, 
telephone 
from dev.naval_silver.customers_cleaned

In [0]:
create or replace view dev.naval_silver.customers_detials as 
select 
customer_id, 
customer_name, 
date_of_birth, 
CASE WHEN
    is_account_group_member('dataeng') THEN email
    ELSE 'REDACTED'
  END AS email,
member_since, 
telephone 
from dev.naval_silver.customers_cleaned

In [0]:
create or replace view dev.naval_silver.customers_detials as 
select 
CASE WHEN
    is_account_group_member('dataeng') THEN customer_id
    ELSE 'REDACTED'
  END AS customer_id, 
customer_name, 
date_of_birth, 
CASE WHEN
    is_account_group_member('dataeng') THEN email
    ELSE 'REDACTED'
  END AS email,
member_since, 
telephone 
from dev.naval_silver.customers_cleaned

Row Level

In [0]:
select * from dev.naval_silver.order_cleaned

customer_id,items,order_date,order_id,order_status,payment_method,total_amount,transaction_timestamp
1987,"List(Map(name -> Drone, item_id -> 10, quantity -> 1, price -> 799, details -> Map(brand -> GoPro, color -> Black), category -> Electronics))",2024-10-28,6,Pending,Credit Card,799,2024-10-28 04:47:27
5816,"List(Map(name -> Tablet, item_id -> 5, quantity -> 1, price -> 399, details -> Map(brand -> GoPro, color -> White), category -> Electronics))",2024-10-07,10,Pending,Credit Card,399,2024-10-07 22:09:27
5816,"List(Map(name -> Smartwatch, item_id -> 4, quantity -> 3, price -> 299, details -> Map(brand -> Canon, color -> White), category -> Electronics))",2024-10-13,11,Pending,Bank Transfer,897,2024-10-13 17:34:19
7207,"List(Map(name -> Smart TV, item_id -> 9, quantity -> 1, price -> 1199, details -> Map(brand -> GoPro, color -> White), category -> Electronics))",2024-10-20,15,Completed,Bank Transfer,1199,2024-10-20 01:47:25
8539,"List(Map(name -> Gaming Console, item_id -> 8, quantity -> 1, price -> 499, details -> Map(brand -> Dell, color -> Silver), category -> Electronics))",2024-10-13,19,Shipped,PayPal,499,2024-10-13 12:40:26
9263,"List(Map(name -> External Hard Drive, item_id -> 7, quantity -> 3, price -> 129, details -> Map(brand -> Sony, color -> Black), category -> Electronics))",2024-10-29,39,Shipped,PayPal,387,2024-10-29 21:39:19
9018,"List(Map(name -> Smartphone, item_id -> 1, quantity -> 1, price -> 699, details -> Map(brand -> HP, color -> Black), category -> Electronics))",2024-10-25,55,Shipped,Bank Transfer,699,2024-10-25 05:49:16
3409,"List(Map(name -> Gaming Console, item_id -> 8, quantity -> 1, price -> 499, details -> Map(brand -> Dell, color -> Black), category -> Electronics))",2024-10-23,59,Pending,PayPal,499,2024-10-23 02:51:05
9084,"List(Map(name -> Wireless Headphones, item_id -> 3, quantity -> 2, price -> 199, details -> Map(brand -> HP, color -> Silver), category -> Electronics))",2024-10-15,77,Pending,Credit Card,398,2024-10-15 13:09:33
2344,"List(Map(name -> Drone, item_id -> 10, quantity -> 1, price -> 799, details -> Map(brand -> Sony, color -> Black), category -> Electronics), Map(name -> Wireless Headphones, item_id -> 3, quantity -> 1, price -> 199, details -> Map(brand -> LG, color -> Black), category -> Electronics))",2024-10-25,84,Completed,PayPal,998,2024-10-25 03:37:42


In [0]:
grant use catalog on catalog dev to `account users`;
grant use schema on schema dev.naval_silver to `account users`;
grant select on table dev.naval_silver.order_transfer to `account users`

In [0]:
create or replace  view dev.naval_silver.order_transfer as 
select * from dev.naval_silver.order_cleaned 
where CASE
    WHEN is_account_group_member('account users') THEN payment_method ='Bank Transfer'
    ELSE True
  END;

In [0]:
-- Step 2: Create view that filters dynamically based on the mapping table
CREATE OR REPLACE VIEW dev.naval_silver.order_transfer AS
SELECT * FROM dev.naval_silver.order_cleaned
WHERE payment_method IN (
  SELECT allowed_payment_method
  FROM dev.naval_silver.access_control ac
  WHERE is_account_group_member(ac.group_name)
);

In [0]:
select * from dev.naval_silver.order_transfer

customer_id,items,order_date,order_id,order_status,payment_method,total_amount,transaction_timestamp
5816,"List(Map(name -> Smartwatch, item_id -> 4, quantity -> 3, price -> 299, details -> Map(brand -> Canon, color -> White), category -> Electronics))",2024-10-13,11,Pending,Bank Transfer,897,2024-10-13 17:34:19
7207,"List(Map(name -> Smart TV, item_id -> 9, quantity -> 1, price -> 1199, details -> Map(brand -> GoPro, color -> White), category -> Electronics))",2024-10-20,15,Completed,Bank Transfer,1199,2024-10-20 01:47:25
9018,"List(Map(name -> Smartphone, item_id -> 1, quantity -> 1, price -> 699, details -> Map(brand -> HP, color -> Black), category -> Electronics))",2024-10-25,55,Shipped,Bank Transfer,699,2024-10-25 05:49:16
5011,"List(Map(name -> Gaming Console, item_id -> 8, quantity -> 2, price -> 499, details -> Map(brand -> GoPro, color -> Black), category -> Electronics))",2024-10-23,85,Shipped,Bank Transfer,998,2024-10-23 20:06:42
3892,"List(Map(name -> Gaming Console, item_id -> 8, quantity -> 2, price -> 499, details -> Map(brand -> Dell, color -> Blue), category -> Electronics))",2024-11-16,5,Cancelled,Bank Transfer,998,2024-11-16 23:07:31
9179,"List(Map(name -> Laptop, item_id -> 2, quantity -> 1, price -> 999, details -> Map(brand -> Apple, color -> Black), category -> Electronics))",2024-11-27,9,Shipped,Bank Transfer,999,2024-11-27 13:15:29
8539,"List(Map(name -> Bluetooth Speaker, item_id -> 6, quantity -> 2, price -> 149, details -> Map(brand -> GoPro, color -> Blue), category -> Electronics))",2024-11-21,21,Cancelled,Bank Transfer,298,2024-11-21 11:25:35
4996,"List(Map(name -> Wireless Headphones, item_id -> 3, quantity -> 1, price -> 199, details -> Map(brand -> GoPro, color -> White), category -> Electronics))",2024-11-12,30,Shipped,Bank Transfer,199,2024-11-12 03:20:29
4914,"List(Map(name -> Wireless Headphones, item_id -> 3, quantity -> 1, price -> 199, details -> Map(brand -> Dell, color -> Gray), category -> Electronics))",2024-11-20,35,Shipped,Bank Transfer,199,2024-11-20 19:09:04
8580,"List(Map(name -> Wireless Headphones, item_id -> 3, quantity -> 1, price -> 199, details -> Map(brand -> HP, color -> Silver), category -> Electronics), Map(name -> External Hard Drive, item_id -> 7, quantity -> 3, price -> 129, details -> Map(brand -> Sony, color -> Silver), category -> Electronics))",2024-11-19,57,Cancelled,Bank Transfer,586,2024-11-19 11:32:44


Data Masking 

In [0]:
CREATE OR REPLACE FUNCTION dev.naval_silver.datamask(x STRING)
  RETURNS STRING
  RETURN CONCAT(REPEAT("*", LENGTH(x) - 2), RIGHT(x, 2)
); 

In [0]:
create or replace view dev.naval_silver.customers_detials_mask as 
select 
customer_id, 
customer_name, 
date_of_birth, 
CASE WHEN
    is_account_group_member('business') THEN dev.naval_silver.datamask(email)
    ELSE email
  END AS email,
member_since, 
telephone 
from dev.naval_silver.customers_cleaned

In [0]:
select * from dev.naval_silver.customers_detials_mask

customer_id,customer_name,date_of_birth,email,member_since,telephone
9179,Richard Cox,1996-10-25,devon84@mail.com,2024-09-26,+1 6680703335
4858,Carla Morton,2004-06-21,joseph88@mail.com,2024-09-15,+1 8616454195
7207,Billy Scott,1997-03-17,christopher30@mail.com,2024-09-23,+1 5544387564
8539,Lori Mason,2002-11-01,stephanie7@mail.com,2024-09-12,+1 0498301620
9706,Jennifer Haas,2001-04-03,benjamin55@mail.com,2024-10-05,+1 4725460000
9263,Joseph Keller,2003-02-11,null,2024-10-04,+1 3817867756
5028,Jessica Harris,2004-04-19,null,2024-09-10,+1 8604009935
9018,William Carter,2003-09-05,james70@gmail.com,2024-10-08,+1 1448753611
8580,Shannon Austin,2002-03-22,john30@gmail.com,2024-10-07,+1 4594705629
3409,Andrew Phillips,2003-04-17,peter73@yahoo.com,2024-09-30,+1 4079273853
